# API QoS Estimation

In [1]:
# First we import the requested modules
import json

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"
pd.set_option("display.precision", 3)

import multiprocessing
from datetime import timedelta

from pandarallel import pandarallel

num_cores = multiprocessing.cpu_count()
pandarallel.initialize()

INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [2]:
# Available colors
colors = [
    "#1f77b4",  # muted blue
    "#ff7f0e",  # safety orange
    "#2ca02c",  # cooked asparagus green
    "#d62728",  # brick red
    "#9467bd",  # muted purple
    "#8c564b",  # chestnut brown
    "#e377c2",  # raspberry yogurt pink
    "#7f7f7f",  # middle gray
    "#bcbd22",  # curry yellow-green
    "#17becf",  # blue-teal
]

In [3]:
# FUNCTIONS
def str_to_int(string):
    final_val = 0
    for c in string:
        val = ord(c)
        final_val += val
    return final_val

In [4]:
f = "../Data/"

In [5]:
# Load line_stops_dict
with open(f + "Static/lines_dict.json") as file:
    lines_dict = json.load(file)

In [6]:
# lines_dict['25']['2']['stops']

## Last week's data

In [7]:
# Read week data — live from SQLite telemetry engine
import os
import sys

nb_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(nb_dir, "..", ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from core import db

week_df = db.get_all_bursts_df("London", None, limit=500_000)
week_df["line"] = week_df["line"].astype(str)
week_df["datetime"] = pd.to_datetime(week_df["datetime"], format="ISO8601")

# Assign direction from destinations
with open(repo_root + "/London/Data/Static/lines_dict.json") as f:
    lines_dict = json.load(f)


def get_dir(row):
    dests = lines_dict.get(str(row["line"]), {}).get("destinations", [])
    if len(dests) > 1 and str(row.get("destination", "")) == dests[1]:
        return 2
    return 1


week_df["direction"] = week_df.apply(get_dir, axis=1)

In [8]:
week_df.head()

,line,bus,vehicleId,destination,stop,estimateArrive,DistanceBus,lat,lon,datetime,direction
0,18,LF75ORC,LF75ORC,Euston,490000091H,1536,7680.0,51.524,-0.143,2026-08-18 11:41:46.376543,1
1,18,LF75OPD,LF75OPD,Euston,490000122E,500,2500.0,51.530,-0.225,2026-08-18 11:41:46.376543,1
2,18,LF75OPS,LF75OPS,Euston,490014975E,100,500.0,51.547,-0.275,2026-08-18 11:41:46.376543,1
3,18,LF75OPK,LF75OPK,Euston,490000011D,764,3820.0,51.522,-0.156,2026-08-18 11:41:46.376543,1
4,18,LF75ONM,LF75ONM,Sudbury,490000252V,118,590.0,51.525,-0.139,2026-08-18 11:41:46.376543,1


In [9]:
week_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   line            500000 non-null  str           
 1   bus             500000 non-null  str           
 2   vehicleId       500000 non-null  str           
 3   destination     500000 non-null  str           
 4   stop            500000 non-null  str           
 5   estimateArrive  500000 non-null  int64         
 6   DistanceBus     500000 non-null  float64       
 7   lat             500000 non-null  float64       
 8   lon             500000 non-null  float64       
 9   datetime        500000 non-null  datetime64[us]
 10  direction       500000 non-null  int64         
dtypes: datetime64[us](1), float64(3), int64(2), str(5)
memory usage: 42.0 MB


# Analysis of temporal series belonging to a bus
Analyze all the data corresponding to the different trips of a bus to a stop.
We pay attention to the last TH values of the series.

In [10]:
# Number of last ocurrences which form the series we are going to analyze for QoS
TH = 30

In [11]:
th_df = week_df.sort_values(by=["bus", "stop", "datetime"], ascending=True)
th_df = th_df.drop_duplicates(["bus", "stop", "datetime"], keep="last")
th_df = th_df[th_df.datetime > th_df.datetime.max() - timedelta(seconds=900)]
th_df.tail(5)

,line,bus,vehicleId,destination,stop,estimateArrive,DistanceBus,lat,lon,datetime,direction
6599,25,SK20BDY,SK20BDY,"City Thameslink, Holborn Viaduct",490015508W1,751,3755.0,51.521,-0.049,2026-08-18 11:39:04.122418,1
5162,25,SK20BDY,SK20BDY,"City Thameslink, Holborn Viaduct",490015508W1,751,3755.0,51.521,-0.049,2026-08-18 11:39:44.658091,1
3726,25,SK20BDY,SK20BDY,"City Thameslink, Holborn Viaduct",490015508W1,709,3545.0,51.521,-0.049,2026-08-18 11:40:25.267916,1
2301,25,SK20BDY,SK20BDY,"City Thameslink, Holborn Viaduct",490015508W1,629,3145.0,51.521,-0.049,2026-08-18 11:41:05.615133,1
856,25,SK20BDY,SK20BDY,"City Thameslink, Holborn Viaduct",490015508W1,589,2945.0,51.521,-0.049,2026-08-18 11:41:46.376543,1


In [12]:
th_df[th_df.line == 18].bus.unique()

<StringArray>
[]
Length: 0, dtype: str

In [13]:
def build_time_series_graph(th_df, TH, bus_id):

    graph = go.Figure()

    # TH_DF
    series_df = th_df[th_df.datetime > th_df.datetime.max() - timedelta(seconds=TH * 30)]

    # Loc Bus Appearances
    series_df = series_df[series_df.bus == bus_id]

    if series_df.shape[0] < 1:
        return graph
    line = series_df.line.iloc[0]
    direction = series_df.direction.iloc[0]
    stops_list = lines_dict[str(line)][str(direction)]["stops"]

    # Set title and layout
    graph.update_layout(
        title=f"<b>Bus {bus_id} : ETA Time Series</b> - Line: {line}",
        legend_title="<b>Destination Stop</b>",
        yaxis={"title": "ETA in Seconds", "nticks": 10, "zerolinecolor": "darkgrey"},
        margin={"r": 0, "l": 0, "t": 40, "b": 0},
        hovermode="closest",
    )

    # Locate unique stops
    unique_stops = series_df.stop.unique().tolist()
    for stop in stops_list:
        if stop not in unique_stops:
            continue
        else:
            stop_index = stops_list.index(stop)

        stop_df = series_df[series_df.stop == stop]

        # Build stop trace
        graph.add_trace(
            go.Scatter(
                name="[" + str(stop_index) + "] " + str(stop),
                x=stop_df.datetime,
                y=stop_df.estimateArrive,
                mode="lines+markers",
                line={"width": 3, "color": colors[(str_to_int(stop)) % len(colors)]},
                text=[
                    "<b>Bus : "
                    + str(bus_id)
                    + "</b> <br>"
                    + "Stop["
                    + str(stop_index)
                    + "]: "
                    + str(stop)
                    + "<br>"
                    + "Time : "
                    + row.datetime.strftime("%H:%M:%S")
                    + "<br>"
                    + "ETA : "
                    + str(row.estimateArrive)
                    for row in stop_df.itertuples()
                ],
                hoverinfo="text",
            )
        )

    return graph

In [14]:
bus_id = "SK20BDV"
build_time_series_graph(th_df, TH, bus_id).show()

In [15]:
bus_id = "BF67GKP"
build_time_series_graph(th_df, TH, bus_id).show()

## Datetime attribute
Should be around actual datetime, and greater than last datetime value

## Estimate Arrive Value Time Series Analysis
- Check the estimations dont follow a linearly descendant curve, it should present some irrugularities, as a straight line indicates the data has just been ponderated making use of the last "trustable" estimation.
- Check the estimations do not present high jumps in their values, this are both changes where the new estimation is much higher or much lower than the previous one.
- Check for continuity, the should be a new approximation every 30 seconds in average, if the time between estimations is higher it means that the API has not provided data for that bus.

In [16]:
# Linearity test
def check_series_linearity(th_df, TH, bus_id):
    # TH_DF
    series_df = th_df[th_df.datetime > th_df.datetime.max() - timedelta(seconds=TH * 30)]

    # Loc Bus Appearances
    series_df = series_df[series_df.bus == bus_id]

    if series_df.shape[0] < 1:
        return {}
    line = series_df.line.iloc[0]
    direction = series_df.direction.iloc[0]
    stops_list = lines_dict[str(line)][str(direction)]["stops"]

    # Slope dict
    slope_dict = {}

    # Locate unique stops
    unique_stops = series_df.stop.unique().tolist()
    for stop in stops_list:
        if stop not in unique_stops:
            continue
        else:
            stops_list.index(stop)

        stop_df = series_df[series_df.stop == stop]

        slope_dict[stop] = {}
        slope_dict[stop]["slopes"] = []
        slope_dict[stop]["quality"] = []
        slope_dict[stop]["time"] = []

        i = 0
        for row in stop_df.itertuples():
            eta = row.estimateArrive
            time = row.datetime

            if i > 0:
                ellapsed_time = int((time - last_time).total_seconds())
                slope = (last_eta - eta) / ellapsed_time

                # Slopes between 0.95 and 1.05 are not good, as they provide no new information, they just ponderate
                # the new estimateArrive value using the last estimation and subtracting the ellapsed sections.
                # On the other hand, slopes of values too high or too low imply changes in the speed of the bus above
                # or below the maximum or minimum speed for a bus, respectively, also implying a bad quality of the
                # estimation received by the API. Therefore, we should penalize both situations.

                # Penalize proximity to 1, and the maximum and lowest possible speeds.

                if (slope < 1) and (slope > 0):
                    slope_quality = (1 - slope) * slope
                elif (slope > 1) and (slope < 2):
                    slope_quality = (2 - slope) * (slope - 1)
                else:
                    slope_quality = 0

                slope_dict[stop]["slopes"].append(slope)
                slope_dict[stop]["quality"].append(slope_quality)
                slope_dict[stop]["time"].append(time)

            i += 1

    return slope_dict

In [17]:
slope_dict = check_series_linearity(th_df, TH, bus_id)

In [18]:
# Draw slopes vs quality series
graph = go.Figure()

# Set title and layout
graph.update_layout(
    title=f"<b>Bus {bus_id} : ETA Time Series</b>",
    legend_title="<b>Destination Stop</b>",
    yaxis={"nticks": 10, "zerolinecolor": "darkgrey"},
    margin={"r": 0, "l": 0, "t": 40, "b": 0},
    hovermode="closest",
)

for stop in slope_dict.keys():
    # Build stop traces
    graph.add_trace(
        go.Scatter(
            name=str(stop) + "-Slope",
            x=slope_dict[stop]["time"],
            y=slope_dict[stop]["slopes"],
            mode="lines+markers",
            line={"width": 3, "color": colors[(str_to_int(stop)) % len(colors)]},
        )
    )

    graph.add_trace(
        go.Scatter(
            name=str(stop) + "-Quality",
            x=slope_dict[stop]["time"],
            y=slope_dict[stop]["quality"],
            mode="lines+markers",
            line={"width": 3, "color": colors[(str_to_int(stop)) % len(colors)]},
        )
    )

graph.show()